# BigQuery Storage Cost analysis

Google charges storage based on either physical bytes or logical bytes.  Here we examine our storage consumption and ask ... which should we use?

Consider a table called `mydataset.mytable`.  This table contains rows of data and hence is consuming storage.  Google requires that we pay for that storage.  However, they give us two billing models ways to pay for the storage (we can choose one or the other).  Those two models are called:

* logical - We pay as a function of the logical storage size.  There is no charge for either time travel or fail safe storage.
* physical - We pay as a function of the physical storage size actually kept on disk.  We also pay for both time travel and fail safe storage.

Imagine a table schema that looks like:

```
name STRING
age INT64
married BOOLEAN
salary FLOAT64
```

The logical storage size is defined as what we count if we summed up the storage data for each column and each row.  This is our intuitive thinking of the size of a table.

When a table is actually written to disk the storage is compressed using a variety of compression techniques.  This can result in signficant compression factors.  Google *always* compresses storage written to disk so do not be tempted to say that "logical storage will give better performance than compressed storage" ... we *always* (under the covers) store using compressed storage.  When the table is written to disk, it occupies some storage amount (bytes) in its compressed form.  With physical billing, Google charges as a function of the compressed storage size.  However, Google will additionally charge for the size of data that is used for time travel and fail safe.

With these notions in mind, we find that *every* table has the following attributes that we can retrieve:

* `ACTIVE_LOGICAL_BYTES` - The amount of logical data consumed by the table that has been modified < 90 days.
* `LONG_TERM_LOGICAL_BYTES` - The amount of logical data consumed by the table that not been modified in the last 90 days.
* `ACTIVE_PHYSICAL_BYTES` - The amount of physical data consumed by the table that has been modified < 90 days.
* `LONG_TERM_PHYSICAL_BYTES` - The amount of logical data consumed by the table that not been modified in the last 90 days.
* `TIME_TRAVEL_PHYSICAL_BYTES` - The amount of physical data maintained for time travel.
* `FAIL_SAFE_PHYSICAL_BYTES` - The amount of physical data maintained for fail safe.

Remember, these are metrics that are kept on every table.  To answer how much we are paying for this storage, we decide if we are paying using the logical model or the physical model.

For the logical model, cost is:

`ACTIVE_LOGICAL_BYTES` * ActiveLogicalStoragePrice + `LONG_TERM_LOGICAL_BYTES` * LongTermLogicalStoragePrice

For the physical model, cost is:

(`ACTIVE_PHYSICAL_BYTES` + `TIME_TRAVEL_PHYSICAL_BYTES` + `FAIL_SAFE_PHYSICAL_BYTES`) * ActivePhysicalStoragePrice + `LONG_TERM_PHYSICAL_BYTES` * LongTermPhysicalStoragePrice

We will charged one or the other.

All tables in a given dataset have to use the same billing model.  Another way of saying this is that the billing model is set at the dataset level and not at the table level.


In [ ]:
# @title Define variables and imports
# @markdown Here we define variables and imports

# Imports needed by this notebook
import pandas
import pandas_gbq

#@markdown The following are tables that are used to access data.  These can name either the real time
#@markdown views or can refer to copies of those tables.
#@markdown
#@markdown ----
#@markdown
#@markdown [INFORMATION_SCHEMA.TABLE_STORAGE_BY_ORGANIZATION](https://cloud.google.com/bigquery/docs/information-schema-table-storage-by-organization)
table_storage_by_organization = "region-us.INFORMATION_SCHEMA.TABLE_STORAGE_BY_ORGANIZATION" # @param ["region-us.INFORMATION_SCHEMA.TABLE_STORAGE_BY_ORGANIZATION"] {"allow-input":true}
#@markdown [INFORMATION_SCHEMA.TABLE_STORAGE_USAGE_TIMELINE_BY_ORGANIZATION](https://cloud.google.com/bigquery/docs/information-schema-table-storage-usage-by-organization)
table_storage_usage_timeline_by_organization = "region-us.INFORMATION_SCHEMA.TABLE_STORAGE_USAGE_TIMELINE_BY_ORGANIZATION" # @param ["region-us.INFORMATION_SCHEMA.TABLE_STORAGE_USAGE_TIMELINE_BY_ORGANIZATION"] {"allow-input":true}


The cost for our storage varies on a variety of factors including:

* Region in which the storage lives
* Is it active Physical
* Is it long-term Physical
* Is it active Logical
* Is it active Physical

Each of these has its own SKU and price per GB per month.  You need to plugin the correct price for accurate results.  You can visit [this](https://cloud.google.com/skus) page to determine the current costs for any given SKUs.  The following are some example SKUs you can use to find prices.

| Region | Active Physical Storage | Active Logical Storage | Long Term Physical Storage | Long Term Logical Storage |
|--|--|--|--|--|
|us|901F-36D9-3005|947D-3B46-7781|2FAA-620A-0686|993F-6B6B-DCC4|
|us-central1|D4B6-F772-0255|0018-A5A0-9D6D|529F-C35B-E521|55B3-2F73-B6A3
|eu|6E26-2D1F-4D81|947D-3B46-7781|A80E-AF0C-6776|993F-6B6B-DCC4|

In [ ]:
#@title Values for storage costs SKUs
active_logical_cost_per_gb_per_month     = 0.02 # @param {"type":"number","placeholder":"Active Logical Cost per Gb/month"}
long_term_logical_cost_per_gb_per_month  = 0.01 # @param {"type":"number","placeholder":"Long term Logical Cost per Gb/month"}
active_physical_cost_per_gb_per_month    = 0.04 # @param {"type":"number","placeholder":"Active physical Cost per Gb/month"}
long_term_physical_cost_per_gb_per_month = 0.02 # @param {"type":"number","placeholder":"Long term Physical Cost per Gb/month"}

In [ ]:
# @title Pipe SQL query to calculate savings/optimizations for storage model by dataset

sql=f"""
DECLARE active_logical_cost_per_gb_per_month     DEFAULT 0.02;
DECLARE long_term_logical_cost_per_gb_per_month  DEFAULT 0.01;
DECLARE active_physical_cost_per_gb_per_month    DEFAULT 0.04;
DECLARE long_term_physical_cost_per_gb_per_month DEFAULT 0.02;
with calc1 AS (
  FROM `{table_storage_by_organization}`
  |> WHERE TABLE_TYPE = "BASE TABLE"
  |> AGGREGATE
       SUM(ACTIVE_LOGICAL_BYTES)/POWER(1024,3)       AS sum_active_logical_gbytes,
       SUM(LONG_TERM_LOGICAL_BYTES)/POWER(1024,3)    AS sum_long_term_logical_gbytes,
       SUM(ACTIVE_PHYSICAL_BYTES - TIME_TRAVEL_PHYSICAL_BYTES)/POWER(1024,3)
                                                     AS sum_active_physical_gbytes,
       SUM(LONG_TERM_PHYSICAL_BYTES)/POWER(1024,3)   AS sum_long_term_physical_gbytes,
       SUM(TIME_TRAVEL_PHYSICAL_BYTES)/POWER(1024,3) AS sum_time_travel_physical_gbytes,
       SUM(FAIL_SAFE_PHYSICAL_BYTES)/POWER(1024,3)   AS sum_fail_safe_physical_gbytes,
       GROUP BY
         PROJECT_ID   as project_id,
         TABLE_SCHEMA as dataset_id
),
storage_types_by_dataset AS (
    FROM `{table_storage_usage_timeline_by_organization}`
    |> WHERE USAGE_DATE = (SELECT DATE_SUB(MAX(USAGE_DATE), INTERVAL 4 DAY) FROM `{table_storage_usage_timeline_by_organization}`)
    |> AGGREGATE SUM(BILLABLE_ACTIVE_LOGICAL_USAGE + BILLABLE_LONG_TERM_LOGICAL_USAGE) AS logical_usage
       GROUP BY
         PROJECT_ID   as project_id,
         TABLE_SCHEMA as dataset_id
    |> EXTEND  CASE
         WHEN logical_usage > 0 THEN 'logical'
         ELSE 'physical'
       END AS current_storage_billing_model
    |> DROP logical_usage
),
results1 AS (
  FROM calc1 LEFT JOIN storage_types_by_dataset ON calc1.project_id = storage_types_by_dataset.project_id AND calc1.dataset_id = storage_types_by_dataset.dataset_id
  |> WHERE storage_types_by_dataset.current_storage_billing_model is not null
  |> SELECT
       calc1.project_id,
       calc1.dataset_id,
       storage_types_by_dataset.current_storage_billing_model,
       ROUND(sum_active_logical_gbytes * active_logical_cost_per_gb_per_month
             + sum_long_term_logical_gbytes * long_term_logical_cost_per_gb_per_month,2) AS logical_cost,
       ROUND((sum_active_physical_gbytes + sum_time_travel_physical_gbytes + sum_fail_safe_physical_gbytes) * active_physical_cost_per_gb_per_month
             + sum_long_term_physical_gbytes * long_term_physical_cost_per_gb_per_month,2) AS physical_cost,
  |> EXTEND ROUND(CASE
       WHEN logical_cost > physical_cost AND current_storage_billing_model = "logical" THEN logical_cost - physical_cost
       WHEN physical_cost > logical_cost AND current_storage_billing_model = "physical" THEN physical_cost - logical_cost
       ELSE 0
     END, 2) as savings
)
SELECT * FROM results1 WHERE savings > 10 ORDER BY savings DESC
"""
pd2 = pandas_gbq.read_gbq(sql)
pd2.head()

Assuming that the above resulted in datasets that could benefit from a change in the current storage billing model, we might now decide to actually change the billing model.

The billing model in effect is an attribute of a dataset and we can change that attribute using:

```
ALTER SCHEMA MY_DATASET SET OPTIONS(storage_billing_model = 'physical')
```

Changing the billing model may take up to 24 hours to take effect.  Additionally, Google prevents us changing the billing model on an individual dataset more frequently that once every 14 days.


The previous calculated which datatsets could be changed to optimize costs.  But what if there were individual tables that could benefit from a model change that were included in a dataset that is *overall* in the best billing model?  The following query is will show individual tables that might benefit from a change.  Likely, it will not show any once the datasets have been optimized.

In [ ]:
# @title SQL query to calculate savings/optimizations for storage model by table
%%bigquery
DECLARE active_logical_cost_per_gb_per_month     DEFAULT 0.02;
DECLARE long_term_logical_cost_per_gb_per_month  DEFAULT 0.01;
DECLARE active_physical_cost_per_gb_per_month    DEFAULT 0.04;
DECLARE long_term_physical_cost_per_gb_per_month DEFAULT 0.02;
with calc1 AS (
  FROM `hy-vee-bigquery-analytics-data.bq_analytics.region_us_table_storage_by_organization`
  |> WHERE TABLE_TYPE = "BASE TABLE"
  |> SELECT ACTIVE_LOGICAL_BYTES/POWER(1024,3)  AS active_logical_gbytes,
       LONG_TERM_LOGICAL_BYTES/POWER(1024,3)    AS long_term_logical_gbytes,
       (ACTIVE_PHYSICAL_BYTES - TIME_TRAVEL_PHYSICAL_BYTES)/POWER(1024,3)
                                                AS active_physical_gbytes,
       LONG_TERM_PHYSICAL_BYTES/POWER(1024,3)   AS long_term_physical_gbytes,
       TIME_TRAVEL_PHYSICAL_BYTES/POWER(1024,3) AS time_travel_physical_gbytes,
       FAIL_SAFE_PHYSICAL_BYTES/POWER(1024,3)   AS fail_safe_physical_gbytes,
       PROJECT_ID                               AS project_id,
       TABLE_SCHEMA                             AS dataset_id,
       TABLE_NAME                               AS table_id
),
storage_types_by_dataset AS (
    FROM `hy-vee-bigquery-analytics-data.bq_analytics.region_us_table_storage_usage_timeline_by_organization`
    |> WHERE USAGE_DATE = (SELECT DATE_SUB(MAX(USAGE_DATE), INTERVAL 4 DAY) FROM `hy-vee-bigquery-analytics-data.bq_analytics.region_us_table_storage_usage_timeline_by_organization`)
    |> AGGREGATE SUM(BILLABLE_ACTIVE_LOGICAL_USAGE + BILLABLE_LONG_TERM_LOGICAL_USAGE) AS logical_usage
       GROUP BY
         PROJECT_ID   as project_id,
         TABLE_SCHEMA as dataset_id
    |> EXTEND  CASE
         WHEN logical_usage > 0 THEN 'logical'
         ELSE 'physical'
       END AS current_storage_billing_model
    |> DROP logical_usage
),
results1 AS (
  FROM calc1 LEFT JOIN storage_types_by_dataset ON calc1.project_id = storage_types_by_dataset.project_id AND calc1.dataset_id = storage_types_by_dataset.dataset_id
  |> WHERE storage_types_by_dataset.current_storage_billing_model is not null
  |> SELECT
      table_id,
       calc1.project_id,
       calc1.dataset_id,
       storage_types_by_dataset.current_storage_billing_model,
       ROUND(active_logical_gbytes * active_logical_cost_per_gb_per_month
             + long_term_logical_gbytes * long_term_logical_cost_per_gb_per_month,2) AS logical_cost,
       ROUND((active_physical_gbytes + time_travel_physical_gbytes + fail_safe_physical_gbytes) * active_physical_cost_per_gb_per_month
             + long_term_physical_gbytes * long_term_physical_cost_per_gb_per_month,2) AS physical_cost,
  |> EXTEND ROUND(CASE
       WHEN logical_cost > physical_cost AND current_storage_billing_model = "logical" THEN logical_cost - physical_cost
       WHEN physical_cost > logical_cost AND current_storage_billing_model = "physical" THEN physical_cost - logical_cost
       ELSE 0
     END, 2) as savings
)
SELECT * FROM results1 WHERE savings > 10 ORDER BY savings DESC

# References
* [Reducing BigQuery physical storage cost with new billing model](https://cloud.google.com/blog/products/data-analytics/new-bigquery-billing-model-helps-reduce-physical-storage-costs)
* [Storage billing models](https://cloud.google.com/bigquery/docs/datasets-intro#dataset_storage_billing_models)
* [BigQuery Storage Billing Models](https://medium.com/qodea/bigquery-storage-billing-models-c2fc48aa8d13)
* [Update storage billing models](https://cloud.google.com/bigquery/docs/updating-datasets#update_storage_billing_models)